# TensorFlow Serving REST/gRPC, wersjonowanie modelu

Notebook demonstruje model server do uruchomienia i utrzymania modelu. Model server pozwala wygodnie zarządzać wersjami modeli produkcyjnie.


In [1]:
! pip install tensorflow tensorflow-serving-api grpcio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 11.3 MB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 11.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 11.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 24.3.25
    Uninstalling flatbuffers-24.3.25:
      Successfully uninstalled flatbuffers-24.3.25
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
  Attempting uninstall: h5py
    Found existing installation: h5py 3.10.0
    Uninstalling h5py-3.10.0:
      Successfully uninstalled h5py-3.10.0
  Attemp

In [2]:
from pathlib import Path
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train[:10000], y_train[:10000], validation_split=0.1, epochs=2, batch_size=128)

export_path = Path("models/mnist/1")
export_path.mkdir(parents=True, exist_ok=True)
model.export(str(export_path))
print(f"SavedModel exported to: {export_path.resolve()}")


I0000 00:00:1781867720.536049     199 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781867720.584153     199 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781867721.877923     199 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


11490434/11490434 [==============================] - 1s 0us/step


W0000 00:00:1781867724.849254     199 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1781867724.857530     199 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1781867724.995635     199 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1781867724.996925     199 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21744 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090 Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 12.0a


Epoch 1/2


I0000 00:00:1781867746.730070     337 service.cc:153] XLA service 0x7b81f8229a20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781867746.730113     337 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5090 Laptop GPU, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.7.1)
I0000 00:00:1781867746.736883     337 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1781867746.773299     337 cuda_dnn.cc:461] Loaded cuDNN version 90701
I0000 00:00:1781867746.836617     337 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


71/71 [==============================] - 22s 6ms/step - loss: 1.0541 - accuracy: 0.7214 - val_loss: 0.4652 - val_accuracy: 0.8790
Epoch 2/2
71/71 [==============================] - 0s 3ms/step - loss: 0.4051 - accuracy: 0.8918 - val_loss: 0.3549 - val_accuracy: 0.9020
INFO:tensorflow:Assets written to: models/mnist/1/assets


INFO:tensorflow:Assets written to: models/mnist/1/assets


Saved artifact at 'models/mnist/1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='input_1')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  135813797602768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135813797603920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135813797603344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135813797603728: TensorSpec(shape=(), dtype=tf.resource, name=None)
SavedModel exported to: /workspace/notebooks/pl/models/mnist/1


## Uruchomienie TensorFlow Serving przez Docker

W terminalu, z katalogu projektu:
```
docker compose
```

komenda dla tensorflow/serving
```bash
docker run -p 8501:8501 -p 8500:8500 \
  --name tfserving_mnist \
  --rm \
  -v "$PWD/models/mnist:/models/mnist" \
  -e MODEL_NAME=mnist \
  tensorflow/serving
```

Port `8501` to REST, a `8500` to gRPC.


In [3]:
import requests

image = x_test[0].tolist()
payload = {"instances": [image]}

response = requests.post(
    "http://host.docker.internal:8501/v1/models/mnist:predict",
    json=payload,
)

print(response.status_code)
print(response.text)


200
{
    "predictions": [[0.000109398, 4.4131607e-06, 0.000287042814, 0.00248835864, 0.000172479238, 0.000141669458, 9.30042643e-06, 0.9918167, 0.000485207973, 0.00448550424]
    ]
}


## Wersjonowanie modelu

Aby dodać nową wersję, wytrenuj model ponownie i zapisz go do katalogu:

```text
models/mnist/2/
```

TensorFlow Serving może obsługiwać nowe wersje modeli bez zmiany kodu klienta.


## gRPC - szkic klienta

gRPC zwykle jest bardziej wydajne niż REST przy komunikacji wewnętrznej między usługami, ale jest mniej wygodne do ręcznego debugowania.


In [4]:
import numpy as np
import grpc
import tensorflow as tf
from tensorflow_serving.apis import prediction_service_pb2_grpc, predict_pb2

channel = grpc.insecure_channel("host.docker.internal:8500")

stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)

request = predict_pb2.PredictRequest()
request.model_spec.name = "mnist"
request.model_spec.signature_name = "serving_default"

image = x_test[0:1].astype(np.float32)  # shape: (1, 28, 28)

request.inputs["input_1"].CopyFrom(
    tf.make_tensor_proto(image, dtype=tf.float32, shape=[1, 28, 28])
)

result = stub.Predict(request, timeout=10.0)
print(result)

outputs {
  key: "output_0"
  value {
    dtype: DT_FLOAT
    tensor_shape {
      dim {
        size: 1
      }
      dim {
        size: 10
      }
    }
    float_val: 0.000109398
    float_val: 4.4131607e-06
    float_val: 0.000287042814
    float_val: 0.00248835864
    float_val: 0.000172479238
    float_val: 0.000141669458
    float_val: 9.30042643e-06
    float_val: 0.9918167
    float_val: 0.000485207973
    float_val: 0.00448550424
  }
}
model_spec {
  name: "mnist"
  version {
    value: 1
  }
  signature_name: "serving_default"
}



In [5]:
print(y_test[0])

7


## Podsumowanie

- Flask/FastAPI są świetne na start.
- TensorFlow Serving jest wyspecjalizowanym model serverem.
- Najważniejsza różnica: wersje modeli, REST/gRPC, canary/A/B testing i mniejszy narzut.
- W produkcji deployment modelu to proces, a nie tylko jeden plik `.h5` albo `.keras`.
